In [1]:
# ============================================================
# CODE BLOCK 1: Clean stable setup for Stage 1 notebook - FIXED
# ============================================================

# Why this block is pinned:
# - transformers==4.44.2 requires huggingface_hub < 1.0.
# - Newer Colab images may already contain gradio 5.x and huggingface_hub 1.x.
# - That combination breaks Transformers 4.44.2.
# - Therefore we uninstall the conflicting packages first and reinstall compatible versions.

!pip uninstall -y huggingface_hub transformers tokenizers gradio

!pip install -q --no-cache-dir \
    huggingface_hub==0.25.2 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    evaluate==0.4.2 \
    peft==0.13.2 \
    bert-score \
    rouge-score \
    scikit-learn \
    pandas \
    tqdm \
    gradio==4.44.1

Found existing installation: huggingface_hub 1.18.0
Uninstalling huggingface_hub-1.18.0:
  Successfully uninstalled huggingface_hub-1.18.0
Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 76.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 248.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 266.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 318.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 456.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 

In [2]:
# IMPORTANT:
# restart the runtime before running Block 2.
# You can do that manually, or run:
# import os
# os.kill(os.getpid(), 9)

In [3]:
# ============================================================
# CODE BLOCK 2: Verify versions and import common libraries
# ============================================================

import os              # Provides functions for interacting with the operating system
                       # (file paths, environment variables, directory operations).

import shutil          # High-level file operations (copy, move, delete directories/files).
                       # Useful for cleaning up output folders between runs.

import torch           # PyTorch library for tensor operations, GPU acceleration,
                       # and deep learning model training.

import numpy as np     # NumPy library for numerical computations, arrays, and
                       # mathematical operations. Often used for metrics and preprocessing.

import pandas as pd    # Pandas library for data manipulation and analysis.
                       # Useful for tabular data inspection or logging results.
import json

import transformers    # Hugging Face Transformers library (models, tokenizers, Trainer).
                       # Core framework for loading and fine-tuning models like CodeGen.

import datasets        # Hugging Face Datasets library for loading and processing datasets.
                       # Provides efficient dataset handling with lazy loading and caching.

import accelerate      # Hugging Face Accelerate library for distributed training,
                       # mixed precision, and hardware abstraction (CPU/GPU/TPU).

import huggingface_hub # Hugging Face Hub client for authentication, uploading models,
                       # and downloading pretrained models/datasets.
import bert_score

from bert_score import score as bert_score

from datasets import load_dataset
# load_dataset is the main function to fetch datasets from Hugging Face Hub.
# Example: load_dataset("code_search_net", "python")

from transformers import AutoTokenizer
# AutoTokenizer automatically loads the correct tokenizer for a given model.
# Tokenizers convert text/code into token IDs that the model can process.

from transformers import AutoModelForCausalLM
# AutoModelForCausalLM loads a causal language model (predicts next token).
# Example: CodeGen, GPT-style models.

from transformers import AutoModel
# AutoModel is a generic loader for models without specifying task type.
# Useful for embeddings or feature extraction.

from transformers import Trainer
# Trainer is Hugging Face’s high-level training loop abstraction.
# Handles training, evaluation, checkpointing, and logging.

from transformers import TrainingArguments
# TrainingArguments defines hyperparameters and settings for Trainer.
# Includes batch size, learning rate, logging, saving, etc.

from transformers import default_data_collator
# default_data_collator batches examples together and pads them to equal length.
# Ensures consistent input shapes for training.

import evaluate
# Hugging Face Evaluate library for metrics (ROUGE, BLEU, CodeBLEU, BERTScore).
# Provides a unified API for computing evaluation scores.

from datasets import concatenate_datasets
# Used to merge Python and Java datasets into one multilingual training dataset.

from peft import LoraConfig
# LoraConfig defines LoRA hyperparameters such as rank, alpha, dropout,
# and which layers should receive LoRA adapters.

from peft import get_peft_model
# get_peft_model injects LoRA adapter layers into the base CodeGen model.

from peft import TaskType
# TaskType tells PEFT what kind of model/task we are adapting.
# Here we use CAUSAL_LM because CodeGen is a decoder-only causal language model.

# Print version numbers to verify environment setup.
print("torch:", torch.__version__)             # Shows installed PyTorch version.
print("transformers:", transformers.__version__) # Shows Transformers version.
print("datasets:", datasets.__version__)         # Shows Datasets version.
print("accelerate:", accelerate.__version__)     # Shows Accelerate version.
print("huggingface_hub:", huggingface_hub.__version__) # Shows Hub client version.

torch: 2.11.0+cu128
transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.34.2
huggingface_hub: 0.25.2


In [4]:
# ============================================================
# CODE BLOCK 3: Configuration
# ============================================================

MODEL_NAME = "Salesforce/codegen-350M-multi"

OFFICIAL_CODESEARCHNET_NAME = "code_search_net"
OFFICIAL_PYTHON_CONFIG = "python"
OFFICIAL_JAVA_CONFIG = "java"

PYTHON_FALLBACK_DATASET_NAME = "Nan-Do/code-search-net-python"
JAVA_FALLBACK_DATASET_NAME = "Nan-Do/code-search-net-java"

MAX_LENGTH = 768

DEMO_MODE = True   # True = quick demo, False = full chunked training

# ------------------------------------------------------------
# Storage strategy
# ------------------------------------------------------------

if DEMO_MODE:
    OUTPUT_DIR = "/content/RepoCoderStudio_Stage1_Demo"
    RESTART_TRAINING = True

    import shutil
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("DEMO MODE: using temporary /content storage.")
    print("No checkpoints will be saved to Google Drive.")

else:
    from google.colab import drive
    drive.mount("/content/drive")

    OUTPUT_DIR = "/content/drive/MyDrive/RepoCoderStudio/Stage1_CodeDocumentation"

    RESTART_TRAINING = False   # Change to True only when intentionally restarting

    if RESTART_TRAINING:
        import shutil
        shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
        print("FULL MODE: output directory cleared. Training will restart.")
    else:
        print("FULL MODE: resume enabled. Existing checkpoints will be reused.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Demo-mode configuration
# ------------------------------------------------------------

PYTHON_TRAIN_SAMPLES = 2500
JAVA_TRAIN_SAMPLES = 2500
PYTHON_EVAL_SAMPLES = 250
JAVA_EVAL_SAMPLES = 250

# ------------------------------------------------------------
# Full-mode chunked training configuration
# ------------------------------------------------------------

NON_DEMO_EVAL_PER_LANGUAGE = 2000
K_SAMPLE_TRAINING = True
K_SAMPLES_PER_LANGUAGE_PER_ROUND = 5000
K_TRAINING_ROUNDS = 80

# ------------------------------------------------------------
# Evaluation configuration
# ------------------------------------------------------------

if DEMO_MODE:
    TEST_EXAMPLES = 50
else:
    TEST_EXAMPLES = 20

# ------------------------------------------------------------
# Reference documentation filtering
# ------------------------------------------------------------
# Training should learn concise function/method documentation, not long API manuals.
# Very long references are filtered out before training/evaluation.

MIN_REFERENCE_DOC_WORDS = 4
MAX_REFERENCE_DOC_WORDS = 150

# ------------------------------------------------------------
# Code length filtering
# ------------------------------------------------------------

MAX_CODE_CHARS = 8000

# ------------------------------------------------------------
# CodeBERTScore length filtering
# ------------------------------------------------------------

MIN_DOC_WORDS_FOR_CODEBERT = 4
MAX_DOC_WORDS_FOR_CODEBERT = 250

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Mode:", "DEMO" if DEMO_MODE else "FULL")
print("Output directory:", OUTPUT_DIR)
print("Evaluation examples:", TEST_EXAMPLES)
print("Reference doc word range:", MIN_REFERENCE_DOC_WORDS, "-", MAX_REFERENCE_DOC_WORDS)
print("Max code chars:", MAX_CODE_CHARS)

DEMO MODE: using temporary /content storage.
No checkpoints will be saved to Google Drive.
Device: cuda
Mode: DEMO
Output directory: /content/RepoCoderStudio_Stage1_Demo
Evaluation examples: 50
Reference doc word range: 4 - 150
Max code chars: 8000


In [5]:
# ============================================================
# CODE BLOCK 4: Load Python + Java documentation datasets
# ============================================================

CODE_FIELD_CANDIDATES = [
    "code",
    "func_code_string",
    "original_string"
]

DOC_FIELD_CANDIDATES = [
    "docstring",
    "func_documentation_string",
    "summary",
    "documentation"
]


def first_available_field(example, candidate_fields):
    """
    Returns the first non-empty field value from a list of possible field names.

    Different CodeSearchNet variants use different column names.
    This helper avoids repeating if/elif logic for every dataset.
    """

    for field in candidate_fields:
        value = example.get(field)

        if value is not None and str(value).strip() != "":
            return value

    return ""


def standardize_dataset_columns(dataset_split, language_name):
    """
    Converts dataset examples into a common Stage 1 schema:

    code
    docstring
    language_tag
    """

    def normalize_example(example):
        return {
            "code": first_available_field(
                example,
                CODE_FIELD_CANDIDATES
            ),
            "docstring": first_available_field(
                example,
                DOC_FIELD_CANDIDATES
            ),
            "language_tag": language_name
        }

    standardized = dataset_split.map(normalize_example)

    standardized = standardized.select_columns(
        ["code", "docstring", "language_tag"]
    )

    return standardized


def get_train_or_first_split(dataset_dict):
    """
    Returns the train split if available.
    Otherwise returns the first available split.

    This keeps the loader robust across official and mirror datasets.
    """

    split_name = "train" if "train" in dataset_dict else list(dataset_dict.keys())[0]

    return dataset_dict[split_name]


def load_official_codesearchnet_split(language_config):
    """
    Loads the official CodeSearchNet split for a specific language.

    Example:
    load_dataset("code_search_net", "python")
    load_dataset("code_search_net", "java")
    """

    return load_dataset(
        OFFICIAL_CODESEARCHNET_NAME,
        language_config
    )


def load_fallback_dataset(fallback_dataset_name):
    """
    Loads the maintained fallback dataset mirror.
    """

    return load_dataset(fallback_dataset_name)


def load_language_dataset(
    language_name,
    official_config,
    fallback_dataset_name
):
    """
    Official-first dataset loading.

    Strategy:
    1. Try official CodeSearchNet language split.
    2. If it fails, log the error.
    3. Load maintained fallback mirror.
    4. Standardize columns into the common Stage 1 schema.
    """

    try:
        print(f"\nTrying official CodeSearchNet {language_name} split...")

        dataset_dict = load_official_codesearchnet_split(
            official_config
        )

        raw_split = get_train_or_first_split(dataset_dict)

        standardized = standardize_dataset_columns(
            raw_split,
            language_name
        )

        print(f"Official CodeSearchNet {language_name} loaded successfully.")
        print(standardized)

        return standardized

    except Exception as official_error:
        print(f"\nOfficial CodeSearchNet {language_name} failed.")
        print("Reason:", official_error)

        print(f"\nFalling back to maintained {language_name} mirror:")
        print(fallback_dataset_name)

        dataset_dict = load_fallback_dataset(
            fallback_dataset_name
        )

        raw_split = get_train_or_first_split(dataset_dict)

        standardized = standardize_dataset_columns(
            raw_split,
            language_name
        )

        print(f"Fallback {language_name} dataset loaded successfully.")
        print(standardized)

        return standardized


# ------------------------------------------------------------
# Load Python and Java datasets
# ------------------------------------------------------------

python_dataset = load_language_dataset(
    language_name="Python",
    official_config=OFFICIAL_PYTHON_CONFIG,
    fallback_dataset_name=PYTHON_FALLBACK_DATASET_NAME
)

java_dataset = load_language_dataset(
    language_name="Java",
    official_config=OFFICIAL_JAVA_CONFIG,
    fallback_dataset_name=JAVA_FALLBACK_DATASET_NAME
)


# ------------------------------------------------------------
# Combine only for inspection/statistics.
# Training split is built later using balanced language sampling.
# ------------------------------------------------------------

combined_dataset = concatenate_datasets(
    [python_dataset, java_dataset]
).shuffle(seed=42)

print("\nCombined multilingual dataset:")
print(combined_dataset)

print("\nCombined columns:")
print(combined_dataset.column_names)

print("\nLanguage distribution:")
print(pd.Series(combined_dataset["language_tag"]).value_counts())


Trying official CodeSearchNet Python split...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:90: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

Map:   0%|          | 0/454451 [00:00<?, ? examples/s]

Official CodeSearchNet Java loaded successfully.
Dataset({
    features: ['code', 'docstring', 'language_tag'],
    num_rows: 454451
})

Combined multilingual dataset:
Dataset({
    features: ['code', 'docstring', 'language_tag'],
    num_rows: 866629
})

Combined columns:
['code', 'docstring', 'language_tag']

Language distribution:
Java      454451
Python    412178
Name: count, dtype: int64


In [6]:
# ============================================================
# CODE BLOCK 5: Multilingual dataset exploration
# ============================================================

print("Combined Dataset Columns:")
print(combined_dataset.column_names)

print("\nPython Sample:")
python_sample = python_dataset[0]
print("Language:", python_sample["language_tag"])
print("Code:\n", python_sample["code"][:1000])
print("\nDocstring:\n", python_sample["docstring"][:500])

print("\nJava Sample:")
java_sample = java_dataset[0]
print("Language:", java_sample["language_tag"])
print("Code:\n", java_sample["code"][:1000])
print("\nDocstring:\n", java_sample["docstring"][:500])

Combined Dataset Columns:
['code', 'docstring', 'language_tag']

Python Sample:
Language: Python
Code:
 def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))
        # logger.debug(str(np.max(seg_border)))
        # logger.debug(str(np.min(seg_border)))
        seg_border[seg_border 

In [7]:
# ============================================================
# CODE BLOCK 6: Prepare multilingual train/eval data
# ============================================================

def safe_select(dataset_obj, start, end):
    """
    Safely selects a slice from a Hugging Face Dataset.

    Why this helper is needed:
    - Python and Java datasets may not have the same size.
    - We should never request indices beyond the dataset length.
    - If start >= end, the function returns None instead of crashing.
    """

    start = min(start, len(dataset_obj))
    end = min(end, len(dataset_obj))

    if start >= end:
        return None

    return dataset_obj.select(range(start, end))


def remove_empty_examples(dataset_obj):
    """
    Removes rows where code or docstring is empty.

    Why this matters:
    - Empty code cannot teach the model anything.
    - Empty docstrings create bad targets.
    - Empty examples can distort training loss and evaluation metrics.
    """

    return dataset_obj.filter(
        lambda example: example["code"].strip() != "" and example["docstring"].strip() != ""
    )


python_dataset = remove_empty_examples(python_dataset)
java_dataset = remove_empty_examples(java_dataset)

print("Python rows after empty filtering:", len(python_dataset))
print("Java rows after empty filtering:", len(java_dataset))


def build_balanced_language_split(
    python_dataset,
    java_dataset,
    python_train_count,
    java_train_count,
    python_eval_count,
    java_eval_count
):
    """
    Builds balanced train/eval splits from Python and Java datasets.

    Why balanced sampling?
    - If Python has many more examples than Java, the model may overfit to Python style.
    - If Java dominates, Python quality may degrade.
    - Balanced sampling gives both languages equal learning opportunity.
    """

    train_parts = []
    eval_parts = []

    python_dataset = python_dataset.shuffle(seed=42)
    java_dataset = java_dataset.shuffle(seed=42)

    python_train = safe_select(
        python_dataset,
        0,
        python_train_count
    )

    python_eval = safe_select(
        python_dataset,
        python_train_count,
        python_train_count + python_eval_count
    )

    java_train = safe_select(
        java_dataset,
        0,
        java_train_count
    )

    java_eval = safe_select(
        java_dataset,
        java_train_count,
        java_train_count + java_eval_count
    )

    if python_train is not None:
        train_parts.append(python_train)

    if java_train is not None:
        train_parts.append(java_train)

    if python_eval is not None:
        eval_parts.append(python_eval)

    if java_eval is not None:
        eval_parts.append(java_eval)

    train_split = concatenate_datasets(train_parts).shuffle(seed=123)
    eval_split = concatenate_datasets(eval_parts).shuffle(seed=456)

    return train_split, eval_split


if DEMO_MODE:
    print("DEMO_MODE=True: using realistic but Colab-friendly multilingual subset.")

    train_data, eval_data = build_balanced_language_split(
        python_dataset=python_dataset,
        java_dataset=java_dataset,
        python_train_count=PYTHON_TRAIN_SAMPLES,
        java_train_count=JAVA_TRAIN_SAMPLES,
        python_eval_count=PYTHON_EVAL_SAMPLES,
        java_eval_count=JAVA_EVAL_SAMPLES
    )

else:
    print("DEMO_MODE=False: preparing data for K-sample chunked training.")

    python_shuffled = python_dataset.shuffle(seed=42)
    java_shuffled = java_dataset.shuffle(seed=42)

    python_eval = safe_select(
        python_shuffled,
        0,
        NON_DEMO_EVAL_PER_LANGUAGE
    )

    java_eval = safe_select(
        java_shuffled,
        0,
        NON_DEMO_EVAL_PER_LANGUAGE
    )

    python_train = safe_select(
        python_shuffled,
        NON_DEMO_EVAL_PER_LANGUAGE,
        len(python_shuffled)
    )

    java_train = safe_select(
        java_shuffled,
        NON_DEMO_EVAL_PER_LANGUAGE,
        len(java_shuffled)
    )

    eval_data = concatenate_datasets(
        [python_eval, java_eval]
    ).shuffle(seed=42)

    train_data = concatenate_datasets(
        [python_train, java_train]
    ).shuffle(seed=42)

print("Train Samples:", len(train_data))
print("Eval Samples:", len(eval_data))

print("\nTrain language distribution:")
print(pd.Series(train_data["language_tag"]).value_counts())

print("\nEval language distribution:")
print(pd.Series(eval_data["language_tag"]).value_counts())

Filter:   0%|          | 0/412178 [00:00<?, ? examples/s]

Filter:   0%|          | 0/454451 [00:00<?, ? examples/s]

Python rows after empty filtering: 412178
Java rows after empty filtering: 454451
DEMO_MODE=True: using realistic but Colab-friendly multilingual subset.
Train Samples: 5000
Eval Samples: 500

Train language distribution:
Java      2500
Python    2500
Name: count, dtype: int64

Eval language distribution:
Java      250
Python    250
Name: count, dtype: int64


In [8]:
# ============================================================
# CODE BLOCK 7: Cleaning helpers and language-aware prompting
# ============================================================

import re

def remove_repeated_sentences(text):
    """
    Removes duplicate repeated sentences.
    Used for both reference documentation and generated documentation.
    """
    if text is None:
        return ""

    text = str(text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)

    unique_sentences = []
    seen = set()

    for sentence in sentences:
        sentence = sentence.strip()
        if sentence == "":
            continue

        normalized = sentence.lower()

        if normalized not in seen:
            unique_sentences.append(sentence)
            seen.add(normalized)

    return " ".join(unique_sentences)


def make_documentation_concise(text, max_sentences=2):
    """
    Keeps documentation concise.
    Stage 1 target is short developer documentation, not long API manuals.
    """
    if text is None:
        return ""

    text = remove_repeated_sentences(text)
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip() != ""
    ]

    return " ".join(sentences[:max_sentences])


def clean_reference_documentation(doc):
    """
    Cleans reference documentation from dataset examples.
    """
    if doc is None:
        return ""

    doc = str(doc)

    # Remove HTML/Javadoc tags
    doc = re.sub(r"<[^>]+>", " ", doc)

    # Remove Javadoc/API sections
    stop_markers = [
        "@param",
        "@return",
        "@throws",
        "@sample",
        "@see"
    ]

    for marker in stop_markers:
        if marker in doc:
            doc = doc.split(marker)[0]

    # Normalize whitespace
    doc = " ".join(doc.split())

    # Remove repeated sentences and keep concise
    doc = make_documentation_concise(
        doc,
        max_sentences=2
    )

    return doc.strip()


def clean_generated_documentation(text):
    """
    Cleans model-generated documentation before evaluation/UI display.
    """
    if text is None:
        return ""

    text = str(text)

    # Remove prompt leakage if present
    if "### Documentation:" in text:
        text = text.split("### Documentation:")[-1]

    stop_markers = [
        "### Task:",
        "### Language:",
        "### Code:",
        "### Inputs:",
        "### Outputs:",
        "def ",
        "\nclass ",
        "\nreturn ",
        "public ",
        "private ",
        "protected ",
        "static ",
        "@Override",
        "@param",
        "@return",
        "@throws",
        "@sample",
        "@see",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    # Remove formatting artifacts
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace('"""', "")
    text = text.replace("/**", "")
    text = text.replace("*/", "")
    text = text.replace("*", "")
    text = text.replace("<|endoftext|>", "")

    # Normalize whitespace
    text = " ".join(text.split())

    # Remove repeated sentences and keep concise
    text = make_documentation_concise(
        text,
        max_sentences=2
    )

    return text.strip()


def get_code(example):
    return example["code"]


def get_doc(example):
    return clean_reference_documentation(
        example["docstring"]
    )


def get_language(example):
    return example["language_tag"]


def build_prompt(code, language):
    """
    Constructs language-aware prompt.
    """
    return (
        "### Task:\n"
        f"Generate concise documentation for the following {language} function or method.\n\n"
        "### Language:\n"
        f"{language}\n\n"
        "### Code:\n"
        f"{code}\n\n"
        "### Documentation:\n"
    )

In [9]:
# ============================================================
# CODE BLOCK 8: Dataset Quality Filtering
# ============================================================

def get_clean_doc_word_count(example):
    cleaned_doc = clean_reference_documentation(
        example["docstring"]
    )
    return len(cleaned_doc.split())


def valid_stage1_example(example):
    """
    Keeps only examples suitable for concise documentation generation.
    """
    code = str(example["code"]).strip()
    cleaned_doc = clean_reference_documentation(
        example["docstring"]
    )

    doc_word_count = len(cleaned_doc.split())

    if code == "":
        return False

    if cleaned_doc == "":
        return False

    if len(code) > MAX_CODE_CHARS:
        return False

    if doc_word_count < MIN_REFERENCE_DOC_WORDS:
        return False

    if doc_word_count > MAX_REFERENCE_DOC_WORDS:
        return False

    return True


def summarize_doc_lengths(dataset_obj, dataset_name):
    word_counts = [
        get_clean_doc_word_count(example)
        for example in dataset_obj.select(
            range(min(1000, len(dataset_obj)))
        )
    ]

    summary = {
        "dataset": dataset_name,
        "sampled_examples": len(word_counts),
        "avg_doc_words": float(np.mean(word_counts)) if word_counts else 0,
        "median_doc_words": float(np.median(word_counts)) if word_counts else 0,
        "max_doc_words": int(np.max(word_counts)) if word_counts else 0,
        "min_doc_words": int(np.min(word_counts)) if word_counts else 0
    }

    print(json.dumps(summary, indent=2))
    return summary


print("\nBefore filtering:")
python_doc_stats_before = summarize_doc_lengths(
    python_dataset,
    "Python before filtering"
)

java_doc_stats_before = summarize_doc_lengths(
    java_dataset,
    "Java before filtering"
)

python_before = len(python_dataset)
java_before = len(java_dataset)

python_dataset = python_dataset.filter(
    valid_stage1_example
)

java_dataset = java_dataset.filter(
    valid_stage1_example
)

python_after = len(python_dataset)
java_after = len(java_dataset)

print("\nAfter filtering:")
python_doc_stats_after = summarize_doc_lengths(
    python_dataset,
    "Python after filtering"
)

java_doc_stats_after = summarize_doc_lengths(
    java_dataset,
    "Java after filtering"
)

print("\nFiltering summary:")
print(f"Python: {python_before} -> {python_after} examples kept")
print(f"Java:   {java_before} -> {java_after} examples kept")

print("\nPython removed:", python_before - python_after)
print("Java removed:", java_before - java_after)

print("\nFiltering complete.")


Before filtering:
{
  "dataset": "Python before filtering",
  "sampled_examples": 1000,
  "avg_doc_words": 16.091,
  "median_doc_words": 12.0,
  "max_doc_words": 101,
  "min_doc_words": 1
}
{
  "dataset": "Java before filtering",
  "sampled_examples": 1000,
  "avg_doc_words": 11.994,
  "median_doc_words": 8.0,
  "max_doc_words": 115,
  "min_doc_words": 0
}


Filter:   0%|          | 0/412178 [00:00<?, ? examples/s]

Filter:   0%|          | 0/454451 [00:00<?, ? examples/s]


After filtering:
{
  "dataset": "Python after filtering",
  "sampled_examples": 1000,
  "avg_doc_words": 16.554,
  "median_doc_words": 13.0,
  "max_doc_words": 101,
  "min_doc_words": 4
}
{
  "dataset": "Java after filtering",
  "sampled_examples": 1000,
  "avg_doc_words": 13.971,
  "median_doc_words": 10.0,
  "max_doc_words": 115,
  "min_doc_words": 4
}

Filtering summary:
Python: 412178 -> 389991 examples kept
Java:   454451 -> 374427 examples kept

Python removed: 22187
Java removed: 80024

Filtering complete.


In [10]:
# ============================================================
# CODE BLOCK 9: Load Baseline Model
# ============================================================

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
# Import classes from Hugging Face Transformers:
# - AutoTokenizer: automatically loads the correct tokenizer for the chosen model.
# - AutoModelForCausalLM: loads a causal language model (predicts next token in sequence).

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME   # Name of the pretrained model defined in CODE BLOCK 3.
                 # Here: "Salesforce/codegen-350M-multi".
                 # Hugging Face will download the tokenizer configuration and vocab.
)

tokenizer.pad_token = tokenizer.eos_token
# Set the padding token to be the same as the end-of-sequence (EOS) token.
# Many causal language models (like CodeGen, GPT) don’t have a dedicated pad token.
# Using EOS ensures consistent padding behavior during batching.

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME   # Load the pretrained causal language model weights.
                 # "Salesforce/codegen-350M-multi" is a 350M parameter model
                 # trained on multiple programming languages.
)

baseline_model.to(DEVICE)
# Move the model to the selected device (GPU if available, otherwise CPU).
# This ensures training and inference run on the correct hardware.


print("Baseline model loaded.")
# Confirmation message to indicate the model and tokenizer are ready

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Baseline model loaded.


In [11]:
# ============================================================
# CODE BLOCK 10: Token Length and Truncation Statistics
# ============================================================

# Purpose:
# This block checks whether MAX_LENGTH=768 is sufficient for the actual
# training and evaluation examples. It does NOT discard data. It only reports
# how often examples are likely to be truncated.

def estimate_prompt_answer_lengths(dataset_obj, sample_size=1000):
    # Limit sample size to dataset length
    sample_size = min(sample_size, len(dataset_obj))
    sampled = dataset_obj.select(range(sample_size))

    rows = []
    for example in sampled:
        # Build prompt text using helper functions.
        # If build_prompt_without_answer exists, use it; otherwise fall back to build_prompt.
        prompt_text = (
            build_prompt_without_answer(example)
            if 'build_prompt_without_answer' in globals()
            else build_prompt(get_code(example), get_language(example))
        )
        # Extract answer (docstring/documentation).
        answer_text = get_doc(example)
        # Concatenate prompt + answer for full sequence.
        full_text = prompt_text + answer_text

        # Tokenize each part without truncation to measure true length.
        prompt_len = len(tokenizer(prompt_text, truncation=False)["input_ids"])
        answer_len = len(tokenizer(answer_text, truncation=False)["input_ids"])
        full_len = len(tokenizer(full_text, truncation=False)["input_ids"])

        # Collect statistics for this example.
        rows.append({
            "language": get_language(example),
            "prompt_tokens": prompt_len,
            "answer_tokens": answer_len,
            "full_tokens": full_len,
            "would_truncate": full_len > MAX_LENGTH
        })

    # Convert collected rows into a DataFrame for analysis.
    stats_df = pd.DataFrame(rows)

    # Compute summary statistics across the sample.
    summary = {
        "sampled_examples": sample_size,
        "max_length": MAX_LENGTH,
        "avg_prompt_tokens": float(stats_df["prompt_tokens"].mean()),
        "avg_answer_tokens": float(stats_df["answer_tokens"].mean()),
        "avg_full_tokens": float(stats_df["full_tokens"].mean()),
        "max_full_tokens": int(stats_df["full_tokens"].max()),
        "truncation_rate_percent": float(stats_df["would_truncate"].mean() * 100),
        "truncation_by_language_percent": stats_df.groupby("language")["would_truncate"].mean().mul(100).to_dict()
    }

    return stats_df, summary

# Run length estimation on evaluation data (up to 1000 examples).
stage1_length_df, stage1_length_summary = estimate_prompt_answer_lengths(
    eval_data,
    sample_size=min(1000, len(eval_data))
)

# Print summary statistics in JSON format for readability.
print("Stage 1 token length summary:")
print(json.dumps(stage1_length_summary, indent=2))

# Save summary statistics to output directory for later inspection.
with open(os.path.join(OUTPUT_DIR, "stage1_token_length_summary.json"), "w") as f:
    json.dump(stage1_length_summary, f, indent=2)


Token indices sequence length is longer than the specified maximum sequence length for this model (2540 > 2048). Running this sequence through the model will result in indexing errors


Stage 1 token length summary:
{
  "sampled_examples": 500,
  "max_length": 768,
  "avg_prompt_tokens": 282.038,
  "avg_answer_tokens": 25.796,
  "avg_full_tokens": 307.834,
  "max_full_tokens": 3786,
  "truncation_rate_percent": 7.199999999999999,
  "truncation_by_language_percent": {
    "Java": 5.2,
    "Python": 9.2
  }
}


In [12]:
# ============================================================
# CODE BLOCK 11: Documentation Generation Helper
# ============================================================

def generate_documentation(model, code, language):
    """
    Generates concise documentation for Python or Java code.
    Uses decoding controls to reduce repetition.
    """

    prompt = build_prompt(
        code,
        language
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,

            # Concise documentation generation
            max_new_tokens=64,

            # Deterministic decoding for evaluation
            do_sample=False,

            # Repetition control
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,

            # Stop and padding behavior
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]

    raw_generated = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    cleaned = clean_generated_documentation(
        raw_generated
    )

    return cleaned.strip()

In [13]:
# ============================================================
# CODE BLOCK 12: Baseline generation helper
# ============================================================

def generate_documentation(model, code, language):
    """
    Generates documentation for either Python or Java code.
    """

    # Build the instruction-style prompt that includes the code and language tag.
    # This prompt tells the model what task to perform (generate documentation).
    prompt = build_prompt(code, language)

    # Tokenize the prompt into input IDs (numerical tokens).
    # - return_tensors="pt" → returns PyTorch tensors.
    # - truncation=True → cuts off text if it exceeds MAX_LENGTH.
    # - max_length=MAX_LENGTH → ensures consistent input size.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move all tensors (input_ids, attention_mask) to the correct device (CPU/GPU).
    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    # Record the length of the input prompt in tokens.
    # This helps us later separate prompt tokens from generated tokens.
    input_length = inputs["input_ids"].shape[1]

    # Put the model in evaluation mode (no dropout, stable inference).
    model.eval()

    # Disable gradient calculations for inference (saves memory, faster).
    with torch.no_grad():
        # Generate new tokens from the model.
        # - max_new_tokens=80 → limit docstring length.
        # - do_sample=False → greedy decoding (deterministic, always pick highest probability).
        # - pad_token_id=tokenizer.eos_token_id → ensures padding handled correctly.
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice off the prompt tokens, keeping only the newly generated documentation tokens.
    generated_tokens = outputs[0][input_length:]

    # Decode token IDs back into human-readable text.
    # - skip_special_tokens=True → removes <pad>, <eos>, etc.
    raw_generated = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Clean the raw text (remove leakage, extra markers) and return final docstring.
    return clean_generated_documentation(raw_generated)

In [14]:
# ============================================================
# CODE BLOCK 13: Baseline example
# ============================================================

sample = eval_data[0]

code = get_code(sample)

reference = get_doc(sample)

language = get_language(sample)

baseline_output = generate_documentation(
    baseline_model,
    code,
    language
)

print("LANGUAGE:\n")
print(language)

print("\nREFERENCE AFTER CLEANING:\n")
print(reference)

print("\nMODEL OUTPUT AFTER CLEANING:\n")
print(baseline_output)
print("\nchecking refernce.split:\n")
len(reference.split())

LANGUAGE:

Java

REFERENCE AFTER CLEANING:

Uploads a server certificate entity for the AWS account. The server certificate entity includes a public key certificate, a private key, and an optional certificate chain, which should all be PEM-encoded.

MODEL OUTPUT AFTER CLEANING:

- (UploadServerCertificateResult) uploadServerCertificate(UploadServerCertificateRequest request) { return executeUploadServerCertificate(request); } ### Response: - (UploadServerCertificateResult) uploadServerCertificate(UploadServerCertificateRequest request, UploadServerCertificateResponse response) { return executeUploadServerCert

checking refernce.split:



31

In [15]:
# ============================================================
# CODE BLOCK 14: Metrics
# ============================================================

import evaluate
# Hugging Face Evaluate library.
# We use it to compute ROUGE-L for documentation similarity.

rouge = evaluate.load("rouge")
# ROUGE-L measures lexical/sequence overlap between generated documentation
# and reference documentation.

In [16]:
# ============================================================
# CODE BLOCK 15: ROUGE-L evaluation function
# ============================================================

def rouge_l_score(prediction, reference):
    """
    Computes ROUGE-L between generated documentation and reference documentation.
    """

    result = rouge.compute(
        predictions=[prediction],
        references=[reference]
    )

    return result["rougeL"]

In [17]:
# ============================================================
# CODE BLOCK 16: CodeBERTScore with length filtering
# ============================================================

# Purpose:
# This block defines helper functions to compute semantic similarity
# between generated documentation and reference documentation using
# CodeBERT (via official BERTScore implementation).
# It includes length filtering to avoid misleading scores from
# empty, very short, or excessively long outputs.

def should_compute_codebert_score(prediction, reference):
    """
    Decides whether CodeBERTScore should be computed.

    Why:
    - Empty outputs should not be embedded (they produce misleading vectors).
    - Very short outputs can give artificially high similarity scores.
    - Very long outputs can slow evaluation and dilute semantic meaning.
    """

    # Split prediction and reference into word lists.
    pred_words = prediction.split()
    ref_words = reference.split()

    # If prediction is too short, skip scoring.
    if len(pred_words) < MIN_DOC_WORDS_FOR_CODEBERT:
        return False

    # If reference is too short, skip scoring.
    if len(ref_words) < MIN_DOC_WORDS_FOR_CODEBERT:
        return False

    # If prediction is too long, skip scoring.
    if len(pred_words) > MAX_DOC_WORDS_FOR_CODEBERT:
        return False

    # If reference is too long, skip scoring.
    if len(ref_words) > MAX_DOC_WORDS_FOR_CODEBERT:
        return False

    # Otherwise, safe to compute CodeBERTScore.
    return True


def codebert_score(prediction, reference):
    """
    Computes semantic similarity using official BERTScore
    with microsoft/codebert-base as the encoder.
    Returns the F1 similarity score as a float.
    """

    # First check if scoring should be skipped due to length constraints.
    if not should_compute_codebert_score(prediction, reference):
        return None

    # Compute BERTScore using CodeBERT as the encoder.
    # - model_type="microsoft/codebert-base" ensures we use CodeBERT.
    # - num_layers=12 matches the full encoder depth.
    # - lang="en" specifies English tokenization.
    # - device=DEVICE ensures GPU/CPU consistency.
    # - verbose=False suppresses extra logging.
    P, R, F1 = bert_score(
        [prediction],
        [reference],
        model_type="microsoft/codebert-base",
        num_layers=12,
        lang="en",
        device=DEVICE,
        verbose=False
    )

    # Return the F1 score as a Python float.
    return float(F1.item())


In [18]:
# ============================================================
# CODE BLOCK 17: Full Stage 1 automatic evaluation
# ============================================================

def evaluate_prediction(prediction, reference):
    """
    Evaluates generated documentation.

    ROUGE-L:
    - Always computed.
    - Empty prediction naturally receives low/zero score.

    CodeBERTScore:
    - Computed only if prediction/reference pass length filter.
    - Otherwise returned as None.
    """

    rouge_score = rouge_l_score(
        prediction,
        reference
    )

    semantic_score = codebert_score(
        prediction,
        reference
    )

    return {
        "ROUGE-L": round(rouge_score, 4),
        "CodeBERTScore": round(semantic_score, 4) if semantic_score is not None else None
    }

In [19]:
# ============================================================
# CODE BLOCK 18: Baseline Evaluation
# ============================================================

TEST_EXAMPLES = min(TEST_EXAMPLES, len(eval_data))

print(f"Running baseline evaluation on {TEST_EXAMPLES} examples...")

baseline_results = []

for i in range(TEST_EXAMPLES):

    example = eval_data[i]

    code = get_code(example)

    reference = get_doc(example)

    language = get_language(example)

    try:
        prediction = generate_documentation(
            baseline_model,
            code,
            language
        )

        # Strict cleaner may return an empty string.
        # We keep that scientifically valid, but label it for readability.
        if prediction.strip() == "":
            prediction = "EMPTY_OUTPUT"

        metrics = evaluate_prediction(
            prediction,
            reference
        )

        baseline_results.append({
            "index": i,
            "language": language,
            "reference": reference,
            "baseline_prediction": prediction,
            "baseline_ROUGE_L": metrics["ROUGE-L"],
            "baseline_CodeBERTScore": metrics["CodeBERTScore"]
        })

    except Exception as error:
        print(f"Error on sample {i}: {error}")

baseline_df = pd.DataFrame(baseline_results)

print("\nCompleted baseline evaluation.")
print("Number of successful evaluations:", len(baseline_df))

baseline_df.head()

Running baseline evaluation on 50 examples...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]


Completed baseline evaluation.
Number of successful evaluations: 50


,index,language,reference,baseline_prediction,baseline_ROUGE_L,baseline_CodeBERTScore
0,0,Java,Uploads a server certificate entity for the AW...,- (UploadServerCertificateResult) uploadServer...,0.0,0.8254
1,1,Python,:param api: :param str vcenter_name: :rtype: V...,EMPTY_OUTPUT,0.0,NaN
2,2,Python,spectrum of correlation matrix and largest cor...,EMPTY_OUTPUT,0.0,NaN
3,3,Java,The notification configurations. NOTE: This me...,### Example:,0.0,NaN
4,4,Python,performs all possible permutations of route im...,EMPTY_OUTPUT,0.0,NaN


In [20]:
# ============================================================
# CODE BLOCK 19: Baseline Metrics Summary
# ============================================================

baseline_avg_rouge = baseline_df["baseline_ROUGE_L"].mean()

baseline_avg_codebert = baseline_df[
    "baseline_CodeBERTScore"
].dropna().mean()

print("=" * 50)
print("BASELINE MODEL PERFORMANCE")
print("=" * 50)

print("Average ROUGE-L:", round(baseline_avg_rouge, 4))
print("Average CodeBERTScore:", round(baseline_avg_codebert, 4))

print("\nBaseline performance by language:")
display(
    baseline_df.groupby("language")[
        ["baseline_ROUGE_L", "baseline_CodeBERTScore"]
    ].mean()
)

print("=" * 50)

BASELINE_ROUGE = baseline_avg_rouge
BASELINE_CODEBERT = baseline_avg_codebert

# ------------------------------------------------------------
# Empty Output Rate
# ------------------------------------------------------------
# Measures how often the model completely failed to generate
# usable documentation.

baseline_empty_outputs = (
    baseline_df["baseline_prediction"] == "EMPTY_OUTPUT"
).sum()

baseline_empty_rate = (
    baseline_empty_outputs / len(baseline_df)
) * 100

print(f"Baseline Empty Output Rate: {baseline_empty_rate:.2f}%")
print(f"Empty Outputs: {baseline_empty_outputs}/{len(baseline_df)}")

BASELINE MODEL PERFORMANCE
Average ROUGE-L: 0.0235
Average CodeBERTScore: 0.7806

Baseline performance by language:


,baseline_ROUGE_L,baseline_CodeBERTScore
language,,
Java,0.020346,0.77525
Python,0.026365,0.79770


Baseline Empty Output Rate: 44.00%
Empty Outputs: 22/50


In [34]:
# ============================================================
# CODE BLOCK 20: Tokenization with docstring-only labels
# ============================================================

def build_prompt_without_answer(example):
    """
    Builds only the input/context portion of the prompt.

    This includes:
    - task instruction
    - programming language
    - source code
    - documentation marker

    It does NOT include the target documentation.

    Why:
    During training, we want the model to use this part as context.
    We do not want to compute loss on this prompt section.
    """

    code = get_code(example)

    language = get_language(example)

    prompt = (
        "### Task:\n"
        f"Generate documentation for the following {language} function or method.\n\n"
        "### Language:\n"
        f"{language}\n\n"
        "### Code:\n"
        f"{code}\n\n"
        "### Documentation:\n"
    )

    return prompt


def build_answer_text(example):
    """
    Builds the target answer portion.

    This is the cleaned reference documentation/docstring.

    Why:
    This is the only part we want the model to learn to generate.
    """

    doc = get_doc(example)

    return doc


def tokenize_for_training(example):
    """
    Tokenizes training examples for causal language model fine-tuning.

    Correct training logic:

    Input sequence:
        prompt + answer

    Labels:
        -100 for prompt tokens
        actual token IDs for answer tokens
        -100 for padding tokens

    Why this is better:
    - The model sees task + language + code as context.
    - Loss is computed only on the documentation.
    - The model is not punished for failing to reproduce the prompt/code.
    - This aligns training with inference, where we provide the prompt and
      expect the model to generate only documentation.
    """

    prompt_text = build_prompt_without_answer(example)

    answer_text = build_answer_text(example)

    full_text = prompt_text + answer_text

    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    tokenized_prompt = tokenizer(
        prompt_text,
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH
    )

    input_ids = tokenized_full["input_ids"]

    attention_mask = tokenized_full["attention_mask"]

    labels = input_ids.copy()

    prompt_length = len(tokenized_prompt["input_ids"])

    labels = [
        token_id if index >= prompt_length and mask == 1 else -100
        for index, (token_id, mask) in enumerate(
            zip(input_ids, attention_mask)
        )
    ]

    tokenized_full["labels"] = labels

    return tokenized_full


if DEMO_MODE:
    tokenized_train = train_data.map(
        tokenize_for_training,
        remove_columns=train_data.column_names
    )

else:
    # In non-demo mode, we avoid tokenizing the full large dataset here.
    # K-sample chunks are tokenized inside Block 23.
    tokenized_train = None


tokenized_eval = eval_data.map(
    tokenize_for_training,
    remove_columns=eval_data.column_names
)

print("Tokenization completed with docstring-only labels.")
print("Eval keys:", tokenized_eval[0].keys())

if tokenized_train is not None:
    print("Train keys:", tokenized_train[0].keys())


# ------------------------------------------------------------
# Sanity check: verify that prompt tokens are masked
# ------------------------------------------------------------

def count_supervised_tokens(tokenized_dataset, n=20):
    """
    Counts how many tokens are actually supervised, i.e.,
    labels not equal to -100.

    This confirms that loss is computed only on documentation tokens.
    """

    supervised_counts = []
    masked_counts = []

    n = min(n, len(tokenized_dataset))

    for i in range(n):
        labels = tokenized_dataset[i]["labels"]

        supervised = sum(
            1 for label in labels if label != -100
        )

        masked = sum(
            1 for label in labels if label == -100
        )

        supervised_counts.append(supervised)
        masked_counts.append(masked)

    return {
        "checked_examples": n,
        "avg_supervised_documentation_tokens": float(np.mean(supervised_counts)),
        "min_supervised_documentation_tokens": int(np.min(supervised_counts)),
        "max_supervised_documentation_tokens": int(np.max(supervised_counts)),
        "avg_masked_prompt_padding_tokens": float(np.mean(masked_counts))
    }


tokenization_sanity = count_supervised_tokens(
    tokenized_eval,
    n=20
)

print("Tokenization sanity check:")
print(json.dumps(tokenization_sanity, indent=2))

Tokenization completed with docstring-only labels.
Eval keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Train keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Tokenization sanity check:
{
  "checked_examples": 20,
  "avg_supervised_documentation_tokens": 22.75,
  "min_supervised_documentation_tokens": 2,
  "max_supervised_documentation_tokens": 78,
  "avg_masked_prompt_padding_tokens": 745.25
}


In [22]:
# ============================================================
# CODE BLOCK 21: Training Example Inspection
# ============================================================

"""
Purpose:
--------
This block is NOT used for training.

It is purely for:
1. Understanding what one training sample looks like.
2. Debugging prompt construction.
3. Demonstrating the training setup.
4. Verifying that the documentation cleaning pipeline works correctly.

Why keep this block?
--------------------
we can show:

Raw code
Reference documentation
Final prompt used for training

This makes the training pipeline transparent and easy to understand.
"""

sample = train_data[0]

print("=" * 80)
print("LANGUAGE")
print("=" * 80)

print(get_language(sample))

print("\n")

print("=" * 80)
print("CODE")
print("=" * 80)

print(get_code(sample)[:1500])

print("\n")

print("=" * 80)
print("REFERENCE DOCUMENTATION (AFTER CLEANING)")
print("=" * 80)

print(get_doc(sample)[:1000])

print("\n")

print("=" * 80)
print("PROMPT PROVIDED TO MODEL")
print("=" * 80)

print(
    build_prompt_without_answer(sample)
)

print("\n")

print("=" * 80)
print("TARGET DOCUMENTATION")
print("=" * 80)

print(
    build_answer_text(sample)
)

print("\n")

print("=" * 80)
print("TRAINING OBJECTIVE")
print("=" * 80)

print(
    "Model sees prompt as context.\n"
    "Loss is computed ONLY on the target documentation.\n"
    "Prompt tokens are masked using -100."
)

LANGUAGE
Java


CODE
public static void scrollToRow(JTable table, int row)
    {
        Rectangle visibleRect = table.getVisibleRect();
        Rectangle cellRect = table.getCellRect(row, 0, true);
        Rectangle r = new Rectangle(
            visibleRect.x, cellRect.y, 
            visibleRect.width, cellRect.height);
        table.scrollRectToVisible(r);
    }


REFERENCE DOCUMENTATION (AFTER CLEANING)
Scroll the given table so that the specified row is visible.


PROMPT PROVIDED TO MODEL
### Task:
Generate documentation for the following Java function or method.

### Language:
Java

### Code:
public static void scrollToRow(JTable table, int row)
    {
        Rectangle visibleRect = table.getVisibleRect();
        Rectangle cellRect = table.getCellRect(row, 0, true);
        Rectangle r = new Rectangle(
            visibleRect.x, cellRect.y, 
            visibleRect.width, cellRect.height);
        table.scrollRectToVisible(r);
    }

### Documentation:



TARGET DOCUMENTATION
S

In [23]:
# ============================================================
# CODE BLOCK 22: Load fresh model and attach LoRA adapters
# ============================================================

try:
    # Delete the previously loaded baseline model (if it exists).
    # This frees up GPU memory before loading a new model.
    del baseline_model
    torch.cuda.empty_cache()  # Clear CUDA cache to avoid OOM errors.
except:
    # If baseline_model was never defined, just skip without crashing.
    pass

# Load the pretrained causal language model (e.g., CodeGen).
# AutoModelForCausalLM automatically selects the right architecture.
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move the model to the correct device (GPU if available, else CPU).
base_model.to(DEVICE)

# Configure LoRA (Low-Rank Adaptation) adapters for parameter-efficient fine-tuning.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Task type: causal language modeling.
    r=8,                            # Rank of low-rank matrices (controls adapter size).
    lora_alpha=16,                  # Scaling factor for LoRA updates.
    lora_dropout=0.05,              # Dropout for regularization (prevents overfitting).
    target_modules=[
        "qkv_proj",                 # Adapt attention query/key/value projection layers.
        "out_proj"                  # Adapt attention output projection layer.
    ],
    bias="none"                     # Do not train bias terms.
)

# Wrap the base model with LoRA adapters.
# This injects small trainable matrices into the specified layers.
finetune_model = get_peft_model(
    base_model,
    lora_config
)

# Print how many parameters are trainable vs frozen.
# Useful sanity check to confirm LoRA is active.
finetune_model.print_trainable_parameters()

# Confirmation message.
print("Fresh CodeGen model loaded with LoRA adapters.")

trainable params: 983,040 || all params: 357,695,488 || trainable%: 0.2748
Fresh CodeGen model loaded with LoRA adapters.


In [24]:
# ============================================================
# CODE BLOCK 23: LoRA fine-tuning setup (updated for full dataset training)
# ============================================================

# Purpose:
# This block defines the Hugging Face TrainingArguments for LoRA fine-tuning.
# It sets up hyperparameters and training behavior for both demo mode and
# full dataset training. Trainer creation is conditional on DEMO_MODE.

# Important:
# - Do NOT delete OUTPUT_DIR here.
# - Restart/resume behavior is controlled only by RESTART_TRAINING in Block 3.

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,              # Directory to save checkpoints/logs.
    num_train_epochs=1,                 # One epoch per chunk (chunked training).
    per_device_train_batch_size=1,      # Small batch size per device.
    per_device_eval_batch_size=1,       # Same for evaluation.
    gradient_accumulation_steps=8,      # Effective batch size = 1 * 8 = 8.
    learning_rate=2e-4,                 # Learning rate tuned for LoRA fine-tuning.
    logging_steps=100,                  # Log metrics every 100 steps.
    save_steps=1000,                    # Save checkpoint every 1000 steps.
    evaluation_strategy="no",           # Disable auto-eval; manual eval in Block 24.
    fp16=False,                         # Disable mixed precision (avoids Colab errors).
    report_to="none"                    # Disable external logging integrations.
)

# Trainer creation depends on DEMO_MODE flag.
if DEMO_MODE:
    # In demo mode, create Trainer immediately with small datasets.
    trainer = Trainer(
        model=finetune_model,           # LoRA-wrapped model to fine-tune.
        args=training_args,             # Training configuration defined above.
        train_dataset=tokenized_train,  # Tokenized training dataset (demo subset).
        eval_dataset=tokenized_eval,    # Tokenized evaluation dataset.
        data_collator=default_data_collator # Handles dynamic padding/batching.
    )
    print("LoRA Trainer ready for demo mode.")
else:
    # In full dataset mode, defer Trainer creation.
    # Trainer will be created later per K-sample chunk (Block 23).
    trainer = None
    print("Non-demo mode: Trainer will be created per K-sample chunk.")


LoRA Trainer ready for demo mode.


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [25]:
# ============================================================
# CODE BLOCK 24: LoRA fine-tuning with K-sample chunked training
# ============================================================

# Purpose:
# This block implements chunked training for large datasets (~800k samples).
# Instead of training on the full dataset at once, it splits data into
# balanced K-sample chunks per language (Python + Java) and trains sequentially.
# Each round saves a checkpoint so training can resume/restart safely.

# Chunk configuration (already defined in Block 3, repeated here for clarity).
K_SAMPLES_PER_LANGUAGE_PER_ROUND = 5000   # Number of samples per language per round
K_TRAINING_ROUNDS = 80                    # Total rounds → ~800k samples

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def select_language_subset(dataset_obj, language_name):
    # Filter dataset by language tag (Python or Java).
    return dataset_obj.filter(lambda example: example["language_tag"] == language_name)

def tokenize_chunk(raw_chunk):
    # Tokenize one chunk using training tokenizer function.
    # Removes original columns to keep only tokenized features.
    return raw_chunk.map(tokenize_for_training, remove_columns=raw_chunk.column_names)

def train_one_chunk(raw_chunk, round_id):
    # Train LoRA adapters on one chunk of data.
    print(f"\nPreparing training chunk for round {round_id}...")
    tokenized_chunk = tokenize_chunk(raw_chunk)

    # Create a new Trainer for this chunk.
    chunk_trainer = Trainer(
        model=finetune_model,
        args=training_args,
        train_dataset=tokenized_chunk,
        eval_dataset=tokenized_eval,
        data_collator=default_data_collator
    )

    print(f"Starting training round {round_id} on {len(tokenized_chunk)} samples...")
    chunk_trainer.train()

    # Save checkpoint for this round.
    checkpoint_path = f"{OUTPUT_DIR}/lora_round_{round_id}"
    finetune_model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)
    print(f"Saved LoRA checkpoint for round {round_id} at: {checkpoint_path}")

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

if DEMO_MODE:
    # In demo mode, train once on demo subset using pre-created Trainer.
    print("DEMO_MODE=True: training once on demo subset.")
    trainer.train()
else:
    # In full dataset mode, use balanced chunked training.
    print("DEMO_MODE=False: using balanced K-sample chunked training.")

    # Separate pools by language.
    python_train_pool = select_language_subset(train_data, "Python")
    java_train_pool = select_language_subset(train_data, "Java")

    # Iterate through training rounds.
    for round_id in range(1, K_TRAINING_ROUNDS + 1):
        checkpoint_path = f"{OUTPUT_DIR}/lora_round_{round_id}"
        # Skip round if checkpoint already exists (resume/restart safety).
        if os.path.exists(checkpoint_path):
            print(f"Checkpoint for round {round_id} exists. Skipping...")
            continue

        round_parts = []

        # Select Python chunk for this round.
        py_start = (round_id - 1) * K_SAMPLES_PER_LANGUAGE_PER_ROUND
        py_end = py_start + K_SAMPLES_PER_LANGUAGE_PER_ROUND
        py_chunk = safe_select(python_train_pool, py_start, py_end)
        if py_chunk is not None:
            round_parts.append(py_chunk)

        # Select Java chunk for this round.
        java_start = (round_id - 1) * K_SAMPLES_PER_LANGUAGE_PER_ROUND
        java_end = java_start + K_SAMPLES_PER_LANGUAGE_PER_ROUND
        java_chunk = safe_select(java_train_pool, java_start, java_end)
        if java_chunk is not None:
            round_parts.append(java_chunk)

        # Stop if no samples remain.
        if len(round_parts) == 0:
            print(f"No more samples for round {round_id}. Stopping.")
            break

        # Concatenate and shuffle chunks for this round.
        raw_round_chunk = concatenate_datasets(round_parts).shuffle(seed=42 + round_id)
        print(f"Round {round_id}: training on {len(raw_round_chunk)} samples.")
        print(pd.Series(raw_round_chunk["language_tag"]).value_counts())

        # Train on this chunk.
        train_one_chunk(raw_round_chunk, round_id)

# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------
print("Fine-tuning completed.")
finetune_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Final LoRA adapter saved to:", OUTPUT_DIR)


DEMO_MODE=True: training once on demo subset.


Step,Training Loss
100,2.143200
200,1.779200
300,1.496100
400,1.644800
500,1.985700
600,1.857500


Fine-tuning completed.
Final LoRA adapter saved to: /content/RepoCoderStudio_Stage1_Demo


In [35]:
# ------------------------------------------------------------
# CODE BLOCK 25: Merge LoRA adapter into the base model already in memory
# ------------------------------------------------------------
from peft import PeftModel

# finetune_model is already your base + adapter
# Merge adapter weights into the base model
merged_model = finetune_model.merge_and_unload()

# Save merged model + tokenizer for direct reuse
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_trained_model"
merged_model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("✅ Final trained model saved at:", FINAL_MODEL_DIR)


✅ Final trained model saved at: /content/RepoCoderStudio_Stage1_Demo/final_trained_model


In [26]:
# ============================================================
# CODE BLOCK 26: Fine-tuned multilingual model evaluation
# ============================================================

TEST_EXAMPLES = min(
    TEST_EXAMPLES,
    len(eval_data)
)

print(f"Running fine-tuned evaluation on {TEST_EXAMPLES} examples...")

finetuned_results = []

for i in range(TEST_EXAMPLES):

    example = eval_data[i]

    code = get_code(example)
    reference = get_doc(example)
    language = get_language(example)

    prediction = generate_documentation(
        finetune_model,
        code,
        language
    )

    if prediction.strip() == "":
        prediction = "EMPTY_OUTPUT"

    metrics = evaluate_prediction(
        prediction,
        reference
    )

    finetuned_results.append({
        "index": i,
        "language": language,
        "reference": reference,
        "finetuned_prediction": prediction,
        "finetuned_ROUGE_L": metrics["ROUGE-L"],
        "finetuned_CodeBERTScore": metrics["CodeBERTScore"]
    })

finetuned_df = pd.DataFrame(
    finetuned_results
)

print("Evaluation completed on", TEST_EXAMPLES, "examples.")
display(finetuned_df.head())

Running fine-tuned evaluation on 50 examples...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Evaluation completed on 50 examples.


,index,language,reference,finetuned_prediction,finetuned_ROUGE_L,finetuned_CodeBERTScore
0,0,Java,Uploads a server certificate entity for the AW...,Uploads a server certificate. This operation c...,0.2273,0.8582
1,1,Python,:param api: :param str vcenter_name: :rtype: V...,:param api: :param str vcenter_name: :rtype: :...,0.4848,0.9250
2,2,Python,spectrum of correlation matrix and largest cor...,spectrum of correlation matrix and largest cor...,0.2979,0.8668
3,3,Java,The notification configurations. NOTE: This me...,The list of notification configurations. Each ...,0.2105,0.7806
4,4,Python,performs all possible permutations of route im...,performs all possible permutations of route im...,1.0000,0.9952


In [27]:
# ============================================================
# CODE BLOCK 27: Fine-tuned average scores
# ============================================================

finetuned_avg_rouge = finetuned_df["finetuned_ROUGE_L"].mean()

finetuned_avg_codebert = finetuned_df[
    "finetuned_CodeBERTScore"
].dropna().mean()

# ------------------------------------------------------------
# Empty Output Rate
# ------------------------------------------------------------
# Measures how often the fine-tuned model failed to generate
# usable documentation.

finetuned_empty_outputs = (
    finetuned_df["finetuned_prediction"] == "EMPTY_OUTPUT"
).sum()

finetuned_empty_rate = (
    finetuned_empty_outputs / len(finetuned_df)
) * 100

print(f"Fine-Tuned Empty Output Rate: {finetuned_empty_rate:.2f}%")
print(
    f"Empty Outputs: "
    f"{finetuned_empty_outputs}/{len(finetuned_df)}"
)

print("Fine-Tuned Average ROUGE-L:", round(finetuned_avg_rouge, 4))
print("Fine-Tuned Average CodeBERTScore:", round(finetuned_avg_codebert, 4))

print("\nFine-tuned performance by language:")
display(
    finetuned_df.groupby("language")[
        ["finetuned_ROUGE_L", "finetuned_CodeBERTScore"]
    ].mean()
)

Fine-Tuned Empty Output Rate: 0.00%
Empty Outputs: 0/50
Fine-Tuned Average ROUGE-L: 0.4388
Fine-Tuned Average CodeBERTScore: 0.908

Fine-tuned performance by language:


,finetuned_ROUGE_L,finetuned_CodeBERTScore
language,,
Java,0.167125,0.855159
Python,0.689608,0.942546


In [28]:
# ============================================================
# CODE BLOCK 28: Comparison table
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Model": "Baseline",
        "ROUGE-L": round(baseline_avg_rouge, 4),
        "CodeBERTScore": round(baseline_avg_codebert, 4),
        "Empty Output Rate (%)": round(
            baseline_empty_rate,
            2
        )
    },
    {
        "Model": "LoRA Fine-Tuned",
        "ROUGE-L": round(finetuned_avg_rouge, 4),
        "CodeBERTScore": round(finetuned_avg_codebert, 4),
        "Empty Output Rate (%)": round(
            finetuned_empty_rate,
            2
        )
    }
])

comparison_df

,Model,ROUGE-L,CodeBERTScore,Empty Output Rate (%)
0,Baseline,0.0235,0.7806,44.0
1,LoRA Fine-Tuned,0.4388,0.9080,0.0


In [31]:
# ============================================================
# CODE BLOCK 29: Example output comparison
# ============================================================

example_comparison_df = baseline_df.merge(
    finetuned_df,
    on=["index", "language"]
)

example_comparison_df[
    [
        "index",
        "language",
        "reference_x",
        "baseline_prediction",
        "finetuned_prediction",
        "baseline_ROUGE_L",
        "finetuned_ROUGE_L",
        "baseline_CodeBERTScore",
        "finetuned_CodeBERTScore"
    ]
].head()

,index,language,reference_x,baseline_prediction,finetuned_prediction,baseline_ROUGE_L,finetuned_ROUGE_L,baseline_CodeBERTScore,finetuned_CodeBERTScore
0,0,Java,Uploads a server certificate entity for the AW...,- (UploadServerCertificateResult) uploadServer...,Uploads a server certificate. This operation c...,0.0,0.2273,0.8254,0.8582
1,1,Python,:param api: :param str vcenter_name: :rtype: V...,EMPTY_OUTPUT,:param api: :param str vcenter_name: :rtype: :...,0.0,0.4848,NaN,0.9250
2,2,Python,spectrum of correlation matrix and largest cor...,EMPTY_OUTPUT,spectrum of correlation matrix and largest cor...,0.0,0.2979,NaN,0.8668
3,3,Java,The notification configurations. NOTE: This me...,### Example:,The list of notification configurations. Each ...,0.0,0.2105,NaN,0.7806
4,4,Python,performs all possible permutations of route im...,EMPTY_OUTPUT,performs all possible permutations of route im...,0.0,1.0000,NaN,0.9952


In [32]:
# ============================================================
# CODE BLOCK 30: Save Results and Experiment Artifacts
# ============================================================

import os
import json
from datetime import datetime

RESULTS_DIR = os.path.join(
    OUTPUT_DIR,
    "evaluation_results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Save detailed evaluation CSVs
# ------------------------------------------------------------

baseline_csv = os.path.join(
    RESULTS_DIR,
    "baseline_results.csv"
)

finetuned_csv = os.path.join(
    RESULTS_DIR,
    "finetuned_results.csv"
)

comparison_csv = os.path.join(
    RESULTS_DIR,
    "comparison_results.csv"
)

examples_csv = os.path.join(
    RESULTS_DIR,
    "example_predictions.csv"
)

baseline_df.to_csv(
    baseline_csv,
    index=False
)

finetuned_df.to_csv(
    finetuned_csv,
    index=False
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)

example_comparison_df.to_csv(
    examples_csv,
    index=False
)

# ------------------------------------------------------------
# Metrics Summary
# ------------------------------------------------------------

metrics_summary = {
    "stage": "Stage 1",
    "task": "Multilingual Code Documentation Generation",

    "model": MODEL_NAME,
    "fine_tuning_method": "LoRA",

    "languages": [
        "Python",
        "Java"
    ],

    "evaluation_examples": int(TEST_EXAMPLES),

    "baseline": {
        "rouge_l": float(round(baseline_avg_rouge, 4)),
        "codebert_score": float(round(baseline_avg_codebert, 4)),
        "empty_output_rate": float(round(
            baseline_empty_rate,
            2
        ))
    },

    "finetuned": {
        "rouge_l": float(round(finetuned_avg_rouge, 4)),
        "codebert_score": float(round(finetuned_avg_codebert, 4)),
        "empty_output_rate": float(round(
            finetuned_empty_rate,
            2
        ))
    },

    "improvements": {
        "rouge_l_gain":
            float(round(
                finetuned_avg_rouge -
                baseline_avg_rouge,
                4
            )),

        "codebert_gain":
            float(round(
                finetuned_avg_codebert -
                baseline_avg_codebert,
                4
            )),

        "empty_output_reduction":
            float(round(
                baseline_empty_rate -
                finetuned_empty_rate,
                2
            ))
    }
}

metrics_json = os.path.join(
    RESULTS_DIR,
    "metrics_summary.json"
)

with open(
    metrics_json,
    "w"
) as f:
    json.dump(
        metrics_summary,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Experiment Card
# ------------------------------------------------------------

experiment_card = {

    "timestamp":
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "stage":
        "Stage 1",

    "objective":
        "Code Documentation Generation",

    "model":
        MODEL_NAME,

    "languages":
        [
            "Python",
            "Java"
        ],

    "max_length":
        MAX_LENGTH,

    "demo_mode":
        DEMO_MODE,

    "lora":
        True,

    "training_mode":
        (
            "Demo"
            if DEMO_MODE
            else "Chunked Full Training"
        ),

    "dataset_filtering": {
        "min_doc_words":
            MIN_REFERENCE_DOC_WORDS,

        "max_doc_words":
            MAX_REFERENCE_DOC_WORDS
    },

    "token_analysis": {
        "max_length":
            MAX_LENGTH,

        "truncation_rate_percent":
            8.4,

        "python_truncation_percent":
            11.6,

        "java_truncation_percent":
            5.2
    },

    "results": metrics_summary
}

experiment_card_path = os.path.join(
    RESULTS_DIR,
    "experiment_card.json"
)

with open(
    experiment_card_path,
    "w"
) as f:
    json.dump(
        experiment_card,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Print Summary
# ------------------------------------------------------------

print("=" * 60)
print("STAGE 1 RESULTS SAVED")
print("=" * 60)

print("\nSaved Files:")

print(baseline_csv)
print(finetuned_csv)
print(comparison_csv)
print(examples_csv)
print(metrics_json)
print(experiment_card_path)

print("\nBaseline:")
print(
    f"ROUGE-L={baseline_avg_rouge:.4f}, "
    f"CodeBERT={baseline_avg_codebert:.4f}, "
    f"Empty={baseline_empty_rate:.2f}%"
)

print("\nFine-Tuned:")
print(
    f"ROUGE-L={finetuned_avg_rouge:.4f}, "
    f"CodeBERT={finetuned_avg_codebert:.4f}, "
    f"Empty={finetuned_empty_rate:.2f}%"
)

print("\nImprovement:")
print(
    f"ROUGE Gain={finetuned_avg_rouge - baseline_avg_rouge:.4f}"
)

print(
    f"CodeBERT Gain={finetuned_avg_codebert - baseline_avg_codebert:.4f}"
)

print(
    f"Empty Output Reduction="
    f"{baseline_empty_rate - finetuned_empty_rate:.2f}%"
)

STAGE 1 RESULTS SAVED

Saved Files:
/content/RepoCoderStudio_Stage1_Demo/evaluation_results/baseline_results.csv
/content/RepoCoderStudio_Stage1_Demo/evaluation_results/finetuned_results.csv
/content/RepoCoderStudio_Stage1_Demo/evaluation_results/comparison_results.csv
/content/RepoCoderStudio_Stage1_Demo/evaluation_results/example_predictions.csv
/content/RepoCoderStudio_Stage1_Demo/evaluation_results/metrics_summary.json
/content/RepoCoderStudio_Stage1_Demo/evaluation_results/experiment_card.json

Baseline:
ROUGE-L=0.0235, CodeBERT=0.7806, Empty=44.00%

Fine-Tuned:
ROUGE-L=0.4388, CodeBERT=0.9080, Empty=0.00%

Improvement:
ROUGE Gain=0.4153
CodeBERT Gain=0.1274
Empty Output Reduction=44.00%


In [33]:
# ============================================================
# CODE BLOCK 31: Colab UI for multilingual Stage 1
# ============================================================

import gradio as gr

def ui_generate_documentation(language, code):
    """
    UI wrapper for multilingual documentation generation.
    Uses the same generation logic as evaluation.
    """

    output = generate_documentation(
        finetune_model,
        code,
        language
    )

    if output.strip() == "":
        output = "EMPTY_OUTPUT"

    return output


example_python_code = """
def factorial(n):
    if n == 0:
        return 1

    result = 1

    for i in range(1, n + 1):
        result *= i

    return result
"""


example_java_code = """
public boolean isPalindrome(String text) {
    String reversed = new StringBuilder(text)
        .reverse()
        .toString();

    return text.equals(reversed);
}
"""


demo = gr.Interface(
    fn=ui_generate_documentation,

    inputs=[
        gr.Dropdown(
            choices=["Python", "Java"],
            value="Python",
            label="Programming Language"
        ),

        gr.Code(
            label="Enter Code",
            language="python",
            value=example_python_code
        )
    ],

    outputs=gr.Textbox(
        label="Generated Documentation",
        lines=5
    ),

    title="RepoCoder Studio - Stage 1 Multilingual Documentation Generator",

    description=(
        "Paste a Python function or Java method. "
        "The LoRA fine-tuned multilingual CodeGen model will generate concise documentation."
    )
)

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://e986da433a7f934f97.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e986da433a7f934f97.gradio.live
